In [1]:
import requests
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', None)

In [2]:
SIMBG_HOST = "https://simbg.pu.go.id"
EMAIL = "dputr@bandungkab.go.id"
PASSWORD = "LogitechG29"

login_resp = requests.post(
    f"{SIMBG_HOST}/api/user/v1/auth/login/",
    json={"email": EMAIL, "password": PASSWORD},
    timeout=30
)
token = login_resp.json()["token"]["access"]
headers = {"Authorization": f"Bearer {token}"}
print("Login OK")

Login OK


In [ ]:
# Fetch semua task tahun 2026
all_data = []
page = 1
while True:
    resp = requests.get(
        # # asli
        # f"{SIMBG_HOST}/api/pbg/v1/list/?page={page}&size=100&sort=ASC&type=task&sort_by=created_at&application_type=1&start_date=2025-01-01&end_date=2026-12-31", 
        # # tugas saya
        # f"{SIMBG_HOST}/api/pbg/v1/list/?page={page}&size=100&sort=ASC&date&search&status&slf_status&type=task&sort_by=created_at&application_type=1&start_date=2025-01-01&end_date=2026-12-31",
        # semua data
        # f"{SIMBG_HOST}/api/pbg/v1/list/?page={page}&size=100&sort=ASC&date&search&status&slf_status&type&sort_by=created_at&application_type=1&start_date=2025-01-01&end_date=2026-12-31",
        f"{SIMBG_HOST}/api/pbg/v1/list/?page={page}&size=100&sort=ASC&date&search&status&type&sort_by=created_at&application_type=1&start_date=2025-01-01&end_date=2026-12-31",
        headers=headers,
        timeout=30
    )
    result = resp.json()
    data = result.get("data", [])
    if not data:
        break
    all_data.extend(data)
    total_pages = result.get("total_page", 1)
    if page >= total_pages:
        break
    page += 1

In [15]:
def get_retribusi(row):
    try:
        r = requests.get(f"{SIMBG_HOST}/api/pbg/v1/detail/{row['uid']}/retribution/submit/", headers=headers, timeout=30)
        data = r.json().get('data', {})
        return data.get('nilai_retribusi_bangunan')
    except:
        pass
    return row['retribution']

df = pd.DataFrame(all_data)
df['retribution'] = df.apply(get_retribusi, axis=1)

In [16]:
print(f"Total rows: {len(df)}")
df.head(10)

Total rows: 2494


,uid,name,owner_name,application_type,application_type_name,condition,registration_number,document_number,address,status,status_name,start_date,slf_status,slf_status_name,function_type,consultation_type,due_date,land_certificate_phase,has_verify_doc,retribution,total_area,unit,slf_expiry,created_at
0,4424a8d1-ba50-44ad-9862-202fb15da0e8,MUHAMAD JAJULI / PT. TOWER BERSAMA,MUHAMAD JAJULI / PT. TOWER BERSAMA,1,Persetujuan Bangunan Gedung,Belum Berdiri,320410-01012025-001,SK-PBG-320410-23012025-002,"Margaasih, Rahayu, Jalan Mahmud Kampung Sindan...",20,SK PBG Terbit,2025-01-01,None,None,Sebagai Bangunan Prasarana,Bangunan Prasarana 1 Lantai,2025-01-24,False,False,9500000.0,25.0,1.0,None,2025-01-01T06:00:56.162850Z
1,41407d29-e9a7-4992-abe1-ec896ff63500,,None,1,Persetujuan Bangunan Gedung,Belum Berdiri,320433-01012025-003,None,"Majalaya, Sukamukti, Kp. Rancawaas RT. 02 RW. ...",3,Permohonan Dibatalkan,2025-01-01,None,None,Sebagai Bangunan Prasarana,Bangunan Prasarana 1 Lantai,2025-08-12,False,False,NaN,36.0,1.0,None,2025-01-01T11:59:56.152306Z
2,a8f7d160-de36-4f22-81fe-21b06c1c3d08,,None,1,Persetujuan Bangunan Gedung,Belum Berdiri,320415-01012025-004,None,"Pangalengan, Sukaluyu, Kp. Mekarmulya RT. 03 R...",3,Permohonan Dibatalkan,2025-01-01,None,None,Sebagai Bangunan Prasarana,Bangunan Prasarana 1 Lantai,2025-08-12,False,False,NaN,100.0,1.0,None,2025-01-01T12:29:05.927538Z
3,b2a22be9-34b6-4e96-915f-1b798e955bee,,None,1,Persetujuan Bangunan Gedung,Belum Berdiri,320408-01012025-005,None,"Bojongsoang, Buahbatu, Kp. Babakan GBI RT. 03 ...",3,Permohonan Dibatalkan,2025-01-01,None,None,Sebagai Bangunan Prasarana,Bangunan Prasarana 1 Lantai,2025-08-12,False,False,NaN,36.0,1.0,None,2025-01-01T12:49:35.266120Z
4,75be256c-375f-472d-98e4-a44008c6244e,wahtoto,Woko Wahtoto an.PT. Tirta Fresindo Jaya,2,Perubahan Bangunan Gedung,Renovasi,320425-02012025-004,SK-PBG-320425-21032025-007,"Cicalengka, Tenjolaya, Jl. Raya Rancaekek Km. ...",20,SK PBG Terbit,2025-01-02,None,None,Fungsi Usaha,Fungsi Usaha 2 lantai,2025-03-24,False,False,623070487.0,50340.0,1.0,None,2024-12-27T00:24:43.572398Z
5,18cd61b1-2d3c-40f5-a2a0-310607864ed6,Reza Wahidy (A.n PT. BINA MITRA SEHATI),Reza Wahidy (A.n PT. BINA MITRA SEHATI),1,Persetujuan Bangunan Gedung,Belum Berdiri,320433-02012025-001,None,"Majalaya, Sukamukti, Kp. Rancawaas RT. 02 RW. ...",16,Retribusi Tidak Sesuai,2025-01-02,None,None,Sebagai Bangunan Prasarana,Bangunan Prasarana 1 Lantai,2025-01-11,False,False,9000000.0,36.0,1.0,None,2025-01-02T00:03:00.889052Z
6,7c1ef191-2e8e-49e0-befc-762cacf09539,Reza Wahidy (A.n PT. BINA MITRA SEHATI),Reza Wahidy (A.n PT. BINA MITRA SEHATI),1,Persetujuan Bangunan Gedung,Belum Berdiri,320415-02012025-002,None,"Pangalengan, Sukaluyu, Kp. Mekarmulya RT. 03 R...",16,Retribusi Tidak Sesuai,2025-01-02,None,None,Sebagai Bangunan Prasarana,Bangunan Prasarana 1 Lantai,2025-01-11,False,False,9000000.0,100.0,1.0,None,2025-01-02T00:16:46.837998Z
7,33d62509-f819-45da-9923-dc893be48e6b,Reza Wahidy (A.n PT. BINA MITRA SEHATI),Reza Wahidy (A.n PT. BINA MITRA SEHATI),1,Persetujuan Bangunan Gedung,Belum Berdiri,320408-02012025-003,None,"Bojongsoang, Buahbatu, Kp. Babakan GBI RT. 03 ...",16,Retribusi Tidak Sesuai,2025-01-02,None,None,Sebagai Bangunan Prasarana,Bangunan Prasarana 1 Lantai,2025-01-11,False,False,9000000.0,36.0,1.0,None,2025-01-02T00:38:08.358991Z
8,eff6c84b-d0bc-42f4-baf0-0c26d69c9e82,"Harry Ramdan / PT Dayamitra Telekomunikasi, Tbk","Harry Ramdan / PT Dayamitra Telekomunikasi, Tbk",1,Persetujuan Bangunan Gedung,Belum Berdiri,320426-02012025-004,SK-PBG-320426-03022025-002,"Nagreg, Citaman, Jl. Lembong No. 62",20,SK PBG Terbit,2025-01-02,None,None,Sebagai Bangunan Prasarana,Bangunan Prasarana 1 Lantai,2025-02-04,False,False,3750000.0,57.0,1.0,None,2025-01-02T05:06:13.192733Z
9,9ce9a635-aca1-4d54-bdb5-cd815c2e4c4f,Luki Lukman Nurhakim,Luki Lukman Nurhakim,1,Persetujuan Bangunan Gedung,Belum Berdiri,320429-06012025-004,SK-PBG-320429-21022025-005,"Ciparay, Cikoneng, Jalan Raya Pacet C

In [18]:
df.retribution.sum()

np.float64(18935112877.5)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2494 entries, 0 to 2493
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   uid                     2494 non-null   object 
 1   name                    2494 non-null   object 
 2   owner_name              2491 non-null   object 
 3   application_type        2494 non-null   object 
 4   application_type_name   2494 non-null   object 
 5   condition               2494 non-null   object 
 6   registration_number     2494 non-null   object 
 7   document_number         1577 non-null   object 
 8   address                 2494 non-null   object 
 9   status                  2494 non-null   int64  
 10  status_name             2494 non-null   object 
 11  start_date              2494 non-null   object 
 12  slf_status              0 non-null      object 
 13  slf_status_name         0 non-null      object 
 14  function_type           2494 non-null   

In [7]:
# PBG data list (dokumen RAB/KRK/DLH)
pbg_data = requests.get(f"{SIMBG_HOST}/api/pbg/v1/task/{uuid}/pbg-data-list/", headers=headers, timeout=30).json()
print(pbg_data)

JSONDecodeError: Expecting value: line 2 column 1 (char 1)

In [8]:
# Detail status (histori status)
detail_status = requests.get(f"{SIMBG_HOST}/api/pbg/v1/task/{uuid}/detail-status/", headers=headers, timeout=30).json()
print(detail_status)

JSONDecodeError: Expecting value: line 2 column 1 (char 1)

In [9]:
# Integrations
integrations = requests.get(f"{SIMBG_HOST}/api/pbg/v1/task/{uuid}/integrations/", headers=headers, timeout=30).json()
print(integrations)

JSONDecodeError: Expecting value: line 2 column 1 (char 1)